# Capítulo 4 · Embeddings y búsqueda semántica

Convertir frases en vectores, medir su similitud, construir un buscador de preguntas frecuentes por significado y compararlo con uno por palabras, trocear un documento largo y dibujar un mapa de significados.

Este cuaderno no usa ningún modelo de lenguaje, solo un modelo de embeddings, así que **funciona bien sin GPU**. Con GPU tarda menos de un minuto; sin ella, unos pocos minutos.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. El modelo y los datos

La primera vez se descargan el modelo de embeddings (algo más de 1 GB), las preguntas frecuentes y las condiciones generales.

In [ ]:
"""Capítulo 4: embeddings y búsqueda semántica. Funciona en GPU o en CPU."""
import json
import re
import unicodedata
import urllib.request
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

MODELO_EMBEDDINGS = "intfloat/multilingual-e5-base"
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/"



def cargar_embeddings(nombre=MODELO_EMBEDDINGS):
    """El modelo de embeddings, en la GPU si la hay."""
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Modelo {nombre} en {dispositivo}")
    return SentenceTransformer(nombre, device=dispositivo)


def descargar(archivo):
    """Un archivo de la carpeta datos; en Colab, se descarga del repositorio."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return ruta.read_text(encoding="utf-8")

In [ ]:
modelo = cargar_embeddings()
datos = json.loads(descargar("preguntas_frecuentes.json"))
documento = descargar("condiciones_generales.md")
print(len(datos["preguntas"]), "preguntas frecuentes,", len(datos["consultas"]), "consultas de prueba")
print(len(documento.split()), "palabras en las condiciones generales")

## 2. Embeddings y similitudes

In [ ]:
def vectorizar(textos, tipo="passage"):
    """Embeddings normalizados. E5 pide «query:» en las preguntas y «passage:» en
    los textos en los que se busca."""
    return modelo.encode([f"{tipo}: {t}" for t in textos],
                         normalize_embeddings=True, convert_to_numpy=True)


def similitudes(frases):
    """Matriz de similitudes del coseno entre unas frases."""
    vectores = vectorizar(frases, "query")
    return vectores @ vectores.T

In [ ]:
v = vectorizar(["¿Cuánto tarda el envío?"], "query")
print(v.shape, "longitud:", np.linalg.norm(v[0]))
print(v[0][:8])

In [ ]:
FRASES = ["¿Cuánto tarda el envío?",
          "¿En cuántos días me llega el paquete?",
          "How long does delivery take?",
          "¿Qué garantía tiene la batería?",
          "Me encanta subir puertos de montaña en bici."]

In [ ]:
for frase, fila in zip(FRASES, similitudes(FRASES)):
    print(f"{frase[:38]:38s}", "  ".join(f"{s:.2f}" for s in fila))

## 3. Buscar por significado y por palabras

In [ ]:
class BuscadorSemantico:
    """Un índice vectorial: una matriz con un embedding por fila."""

    def __init__(self, textos):
        self.textos = textos
        self.matriz = vectorizar(textos, "passage")

    def buscar(self, consulta, k=3):
        puntos = self.matriz @ vectorizar([consulta], "query")[0]
        mejores = np.argsort(-puntos)[:k]
        return [(int(i), float(puntos[i])) for i in mejores]

In [ ]:
PALABRAS_VACIAS = set(
    "a al algo con de del el en es la las lo los me mi mis no o para por que se "
    "si su sus te tu un una unos y ya yo os vuestro vuestra hay le les cual como "
    "cuando donde qué cómo cuál dónde cuánto cuánta cuántos cuántas".split())


def sin_tildes(texto):
    """Minúsculas y sin tildes."""
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower())
                   if unicodedata.category(c) != "Mn")


class BuscadorPalabras:
    """Búsqueda clásica por palabras con TF-IDF: cuenta palabras compartidas,
    dando más peso a las que aparecen en pocos textos."""

    def __init__(self, textos):
        self.textos = textos
        vacias = sorted({sin_tildes(p) for p in PALABRAS_VACIAS})
        self.tfidf = TfidfVectorizer(preprocessor=sin_tildes, stop_words=vacias)
        self.matriz = self.tfidf.fit_transform(textos)

    def buscar(self, consulta, k=3):
        puntos = (self.matriz @ self.tfidf.transform([consulta]).T).toarray()[:, 0]
        mejores = np.argsort(-puntos)[:k]
        return [(int(i), float(puntos[i])) for i in mejores]

In [ ]:
textos = [p["pregunta"] + " " + p["respuesta"] for p in datos["preguntas"]]
semantico, palabras = BuscadorSemantico(textos), BuscadorPalabras(textos)
consulta = "¿Dónde puedo ver por dónde va mi paquete?"
for nombre, buscador in (("Significado", semantico), ("Palabras", palabras)):
    print(nombre)
    for i, puntos in buscador.buscar(consulta):
        print(f"   {puntos:.2f}  {datos['preguntas'][i]['pregunta']}")

In [ ]:
def evaluar_faq(buscador, consultas):
    """Aciertos en la primera posición y entre las tres primeras."""
    en_1 = en_3 = 0
    for consulta, correcta in consultas:
        posiciones = [i + 1 for i, _ in buscador.buscar(consulta, k=3)]
        en_1 += posiciones[0] == correcta
        en_3 += correcta in posiciones
    return en_1, en_3


def probar_faq(datos):
    """Buscador de preguntas frecuentes: por significado frente a por palabras."""
    textos = [p["pregunta"] + " " + p["respuesta"] for p in datos["preguntas"]]
    resultados = {}
    for nombre, clase in (("palabras", BuscadorPalabras),
                          ("significado", BuscadorSemantico)):
        en_1, en_3 = evaluar_faq(clase(textos), datos["consultas"])
        resultados[nombre] = (en_1, en_3)
        print(f"Por {nombre:11s}: primera posición {en_1}/30, "
              f"entre las tres primeras {en_3}/30")
    return resultados

In [ ]:
faq = probar_faq(datos)

## 4. Trocear un documento largo

In [ ]:
def secciones(documento):
    """Un fragmento por sección del documento (las que empiezan por ##)."""
    return [s.strip() for s in re.split(r"\n(?=## )", documento) if s.startswith("##")]


def trocear(documento, palabras, solape=0):
    """Fragmentos de un número fijo de palabras; cada uno repite las últimas
    «solape» palabras del anterior."""
    texto = re.sub(r"#+ ", "", documento).split()
    paso = palabras - solape
    return [" ".join(texto[i:i + palabras])
            for i in range(0, max(len(texto) - solape, 1), paso)]

In [ ]:
fragmentos = trocear(documento, 40)
print(len(fragmentos), "fragmentos de 40 palabras. Los dos primeros:")
print(fragmentos[0], "\n---\n", fragmentos[1])

In [ ]:
PREGUNTAS_CONDICIONES = [
    ("¿Puedo cancelar un pedido que ya he pagado?", "se puede cancelar sin coste"),
    ("Si pago por transferencia, ¿cuánto tiempo tengo?", "plazo de cinco días"),
    ("¿Guardáis los números de mi tarjeta?", "no almacena en ningún momento"),
    ("¿Qué pasa si no estoy en casa cuando llega el repartidor?",
     "lo intentará una segunda vez"),
    ("La caja llegó golpeada, ¿qué tengo que hacer?", "anotarlo en el albarán"),
    ("¿Puedo devolver un culote que ya he abierto?",
     "no se admiten devoluciones de culotes"),
    ("¿Me devolvéis también lo que pagué de envío?",
     "gastos del envío original solo se reembolsan"),
    ("¿Puedo cambiar en la tienda unas zapatillas compradas por internet?",
     "no hacemos cambios directos en la tienda"),
    ("¿Qué capacidad debe mantener la batería durante la garantía?",
     "al menos el 70 %"),
    ("¿Qué incluye la revisión de 35 euros?", "ajuste de cambios y frenos"),
    ("¿Cómo pido que borréis mis datos?", "privacidad@ciclosnorte.es"),
    ("¿Cuánto tardáis en contestar los correos?", "plazo máximo de 48 horas"),
    ("Si un producto se agota después de pedirlo, ¿qué pasa?",
     "avisaremos al cliente en un plazo de 24 horas"),
    ("¿Los precios llevan el IVA incluido?", "incluyen el iva"),
    ("Desde Canarias, ¿quién paga el envío de la devolución?",
     "desde baleares y canarias, el coste del envío de vuelta"),
]


def evaluar_troceado(fragmentos, preguntas=PREGUNTAS_CONDICIONES):
    """¿Contiene el fragmento recuperado la frase que responde a la pregunta?"""
    buscador = BuscadorSemantico(fragmentos)
    en_1 = en_3 = 0
    for pregunta, frase in preguntas:
        recuperados = [fragmentos[i].lower() for i, _ in buscador.buscar(pregunta, 3)]
        en_1 += frase in recuperados[0]
        en_3 += any(frase in f for f in recuperados)
    return en_1, en_3


def probar_troceados(documento):
    """Distintas formas de trocear las condiciones generales."""
    variantes = {"Por secciones": secciones(documento)}
    for palabras in (40, 80, 160, 320):
        for solape in (0, palabras // 4):
            variantes[f"{palabras} palabras, solape {solape}"] = trocear(
                documento, palabras, solape)
    resultados = {}
    for nombre, fragmentos in variantes.items():
        en_1, en_3 = evaluar_troceado(fragmentos)
        resultados[nombre] = (len(fragmentos), en_1, en_3)
        print(f"{nombre:28s} {len(fragmentos):3d} fragmentos  "
              f"primero {en_1:2d}/15  tres primeros {en_3:2d}/15")
    return resultados

In [ ]:
troceados = probar_troceados(documento)

## 5. Un mapa de significados

In [ ]:
def mapa(datos):
    """Coordenadas 2D de las preguntas frecuentes, con PCA."""
    vectores = vectorizar([p["pregunta"] for p in datos["preguntas"]], "query")
    return PCA(n_components=2).fit_transform(vectores)

In [ ]:
import matplotlib.pyplot as plt

xy = mapa(datos)
temas = sorted({p["tema"] for p in datos["preguntas"]})
plt.figure(figsize=(8, 5.5))
for tema in temas:
    puntos = [(x, y, p["id"]) for (x, y), p in zip(xy, datos["preguntas"]) if p["tema"] == tema]
    plt.scatter([p[0] for p in puntos], [p[1] for p in puntos], label=tema)
    for x, y, i in puntos:
        plt.annotate(str(i), (x, y), xytext=(4, 3), textcoords="offset points", fontsize=7)
plt.legend()
plt.show()

## Ejercicios

1. Explica con tus palabras qué es un embedding de frase y por qué dos frases sin palabras en común pueden tener embeddings cercanos.

2. Calcula a mano la similitud del coseno entre los vectores (1, 2, 2) y (2, 0, 1). Normaliza los dos vectores y comprueba que su producto escalar da el mismo resultado.

3. Añade a `FRASES` una frase que diga lo contrario de la primera, como «El envío es inmediato, no tarda nada». ¿Qué similitud tiene con «¿Cuánto tarda el envío?»? ¿Qué te dice eso sobre lo que capturan los embeddings?

4. Escribe diez consultas que usen las mismas palabras que las preguntas frecuentes y repite la comparación entre los dos buscadores. ¿Cuál gana ahora?

5. Escribe tres consultas con un número de pedido o un modelo concreto, como «¿Ha salido ya el CN-10482?», y añade a las preguntas frecuentes un texto que lo contenga. ¿Qué buscador lo encuentra?

6. Quita los prefijos `query:` y `passage:` de `vectorizar`. ¿Cambian los aciertos?

7. Cambia `MODELO_EMBEDDINGS` por `intfloat/multilingual-e5-small` y repite todo. ¿Cuánto pierdes en aciertos y cuánto ganas en tiempo?

8. Combina secciones y tamaño fijo: trocea por secciones y, si una sección tiene más de 120 palabras, pártela en fragmentos de 80 con un solape de 20. Compara con la tabla.

9. Sustituye PCA por t-SNE (`sklearn.manifold.TSNE`, con `perplexity=5`) y dibuja de nuevo el mapa. ¿Se separan mejor los temas? ¿Qué inconvenientes tiene t-SNE?

10. **Buscador interactivo.** Escribe una función que reciba una consulta y muestre las tres preguntas frecuentes más parecidas con su respuesta y su similitud. Pruébala con diez consultas tuyas y anota en cuáles falla. ¿Encuentras algún patrón?

## Referencias

Modelo [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base), de Microsoft, con licencia MIT. [Sentence-BERT](https://arxiv.org/abs/1908.10084) (Reimers y Gurevych, 2019), [E5](https://arxiv.org/abs/2212.03533) (Wang y colaboradores, 2022) y [MTEB](https://arxiv.org/abs/2210.07316) (Muennighoff y colaboradores, 2022). Lección 11.04 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La tienda Ciclos Norte, sus preguntas frecuentes y sus condiciones son ficticias. Código y experimentos propios.